# Stage 9 — Model Training and Cross-Validation

Cross-validation is performed on training countries only. Because the sample is small, CV results should be interpreted as approximate rather than definitive.

The country-level design produces fewer than 200 observations overall. With a 75/25 split, each five-fold CV validation fold contains only a small number of countries. This limitation is especially relevant for flexible ensemble models such as Random Forest and Extra Trees, which can fit complex patterns when the number of predictors is high relative to the sample size. The notebook therefore uses modest tuning, restricts the final predictor count, reports test-set bootstrap intervals, and avoids causal interpretation.

## 9.1 Cross-validation setup

In [379]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 9.2 Train and compare models using training CV

In [381]:
def evaluate_predictions(y_true_log, y_pred_log):
    return {
        "rmse_log": root_mean_squared_error(y_true_log, y_pred_log),
        "mae_log": mean_absolute_error(y_true_log, y_pred_log),
        "r2_log": r2_score(y_true_log, y_pred_log),
        "mae_original_scale": mean_absolute_error(np.exp(y_true_log), np.exp(y_pred_log))
    }

cv_rows = []
fitted_initial_models = {}

for model_name, pipeline in model_catalogue.items():
    cv_scores = -cross_val_score(
        pipeline,
        X_train,
        y_train,
        scoring="neg_root_mean_squared_error",
        cv=cv,
        n_jobs=-1
    )
    pipeline.fit(X_train, y_train)
    fitted_initial_models[model_name] = pipeline

    cv_rows.append({
        "model": model_name,
        "cv_rmse_log_mean": cv_scores.mean(),
        "cv_rmse_log_std": cv_scores.std()
    })

cv_results = pd.DataFrame(cv_rows).sort_values("cv_rmse_log_mean")
cv_results.to_csv(TABLE_DIR / "cv_model_comparison_train_only.csv", index=False)
display(cv_results)

best_cv_model_name = cv_results[cv_results["model"] != "Dummy mean"].iloc[0]["model"]
print("Best non-dummy model by training CV:", best_cv_model_name)

,model,cv_rmse_log_mean,cv_rmse_log_std
5,Extra Trees,0.438749,0.136656
6,Gradient Boosting,0.464387,0.105211
7,XGBoost,0.478287,0.117580
4,Random Forest,0.495455,0.109599
3,KNN,0.521236,0.070717
1,Ridge,0.555604,0.138292
2,ElasticNet,0.809528,0.186305
0,Dummy mean,1.039777,0.177150


Best non-dummy model by training CV: Extra Trees
